<a href="https://colab.research.google.com/github/anant060806/GPU_/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!nvcc --version


Mon Aug 31 16:24:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**SIMT IS USED IN GPU **
-> THREADS : lowest granuality of excution

-> WARPS : lowest schedulable entity
            ->not every thread needs to excute all instruction

->thread blocks: lowest programabe entity,assigned to single
                 shader core
                 can be three dimentional

  ->grids: group of thread blocks, how the problem is matched in gpu
                  a grid is defined as the highest-level organizational unit that contains all the thread blocks launched to execute a single kernel function.
                  grid is bound to a single compiled kernel function.



In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter


The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [ ]:

#include <iostream>

__global__void helloFromGPU() {
    printf("Hello World from GPU thread %d!\n", threadIdx.x);
}

int main() {
    std::cout << "Hello World from CPU!" << std::endl;
    helloFromGPU<<<1, 5>>>();
    cudaDeviceSynchronize();
    return 0;
}


SyntaxError: invalid syntax (3677067459.py, line 3)

In [ ]:
!nvidia-smi


Fri Sep 18 14:17:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile add.cu
#include<iostream>
__global__voidvectoradd(int*a,int*b,int*c,n){
    int tid=(blockIdx.x*blockDim.x)+threadIdx.x;
    if(tid<n){
        c[tid]=a[tid]+b[tid];
    }
}

Writing add.cu


In [ ]:
%%writefile vector_add.cu
# This program computes the sum of two vectors of length N

#include <algorithm>
#include <cassert>
#include <iostream>
#include <vector>

# CUDA kernel for vector addition
#_global__ means this is called from the CPU, and runs on the GPU
__global__ void vectorAdd(const int *__restrict a, const int *__restrict b,
                          int *__restrict c, int N) {
  # Calculate global thread ID
  int tid = (blockIdx.x * blockDim.x) + threadIdx.x;

  # Boundary check
  if (tid < N) c[tid] = a[tid] + b[tid];
}

# Check vector add result
void verify_result(std::vector<int> &a, std::vector<int> &b,
                   std::vector<int> &c) {
  for (int i = 0; i < a.size(); i++) {
    assert(c[i] == a[i] + b[i]);
  }
}

int main() {
  # Array size of 2^16 (65536 elements)
  constexpr int N = 1 << 16;
  constexpr size_t bytes = sizeof(int) * N;

  ## Vectors for holding the host-side (CPU-side) data
  std::vector<int> a;
  a.reserve(N);
  std::vector<int> b;
  b.reserve(N);
  std::vector<int> c;
  c.reserve(N);

  #Initialize random numbers in each array
  for (int i = 0; i < N; i++) {
    a.push_back(rand() % 100);
    b.push_back(rand() % 100);
  }

  #Allocate memory on the device
  int *d_a, *d_b, *d_c;
  cudaMalloc(&d_a, bytes);
  cudaMalloc(&d_b, bytes);
  cudaMalloc(&d_c, bytes);


  cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice);

  # Threads per CTA (1024)
  int NUM_THREADS = 1 << 10;


  int NUM_BLOCKS = (N + NUM_THREADS - 1) / NUM_THREADS;


  vectorAdd<<<NUM_BLOCKS, NUM_THREADS>>>(d_a, d_b, d_c, N);

  cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost);


  verify_result(a, b, c);


  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);

  std::cout << "COMPLETED SUCCESSFULLY\n";

  return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
vector_add.cu:1:3: error: invalid preprocessing directive #This
    1 | # This program computes the sum of two vectors of length N
      |   ^~~~
vector_add.cu:8:3: error: invalid preprocessing directive #CUDA
    8 | # CUDA kernel for vector addition
      |   ^~~~
vector_add.cu:9:2: error: invalid preprocessing directive #_global__
    9 | #_global__ means this is called from the CPU, and runs on the GPU
      |  ^~~~~~~~~
vector_add.cu:12:5: error: invalid preprocessing directive #Calculate
   12 |   # Calculate global thread ID
      |     ^~~~~~~~~
vector_add.cu:15:5: error: invalid preprocessing directive #Boundary
   15 |   # Boundary check
      |     ^~~~~~~~
vector_add.cu:19:3: error: invalid preprocessing directive #Check
   19 | # Check vector add result
      |   ^~~~~
vector_add.cu:28:5: 